In [ ]:
#This code generates Figure 6(a) of the manuscript

import numpy as np
import matplotlib.pyplot as plt
import torch
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(123456)
np.random.seed(123456)

def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.xavier_normal_(m.weight)
        nn.init.constant_(m.bias, 0.0)        
class MDN(nn.Module):
    def __init__(self, n_hidden, n_gaussians,n_hidden_layers,bn, dout, dout_value):
        super(MDN, self).__init__()
        layers = [nn.Linear(41, n_hidden), nn.Tanh()] 
        if bn == 1:
            layers.append(nn.BatchNorm1d(n_hidden))    
        for _ in range(n_hidden_layers - 2):
            layers.append(nn.Linear(n_hidden, n_hidden))
            layers.append(nn.Tanh())
        if dout==1:
            layers.append(nn.Dropout(dout_value))
            layers.append(nn.Linear(n_hidden, n_hidden))
            layers.append(nn.Tanh())   
        else:
            layers.append(nn.Linear(n_hidden, n_hidden))
            layers.append(nn.Tanh())       
        self.z_h = nn.Sequential(*layers)
        self.z_pi = nn.Linear(n_hidden, n_gaussians)
        self.z_sigma = nn.Linear(n_hidden, n_gaussians)
        self.z_mu = nn.Linear(n_hidden, n_gaussians)  
    def forward(self, x):
        z_h = self.z_h(x)
        pi = nn.functional.softmax(self.z_pi(z_h), -1)
        sigma = torch.exp(self.z_sigma(z_h))+ 1e-8
        sigma = torch.clamp(sigma, min=1e-4)
        mu = 0 + (1 - 0) * (torch.tanh(self.z_mu(z_h)) + 1) / 2
        return pi, sigma, mu
oneDivSqrtTwoPI = 1.0 / np.sqrt(2.0*np.pi) 
def gaussian_distribution(y, mu, sigma):
    result = (y.expand_as(mu) - mu) * torch.reciprocal(sigma)
    result = -0.5 * (result * result)
    return (torch.exp(result) * torch.reciprocal(sigma)) * oneDivSqrtTwoPI
def mdn_loss_fn(pi, sigma, mu, y):
    result = gaussian_distribution(y, mu, sigma) * pi
    result1 = torch.sum(result, dim=1)
    result2 = -torch.log(result1+1e-12)
    return result2
def compute_log_likelihood(mdn_model, data_loader, device):
    mdn_model.eval()
    total_log_likelihood = 0.0
    with torch.no_grad():
        for x, y in data_loader:
            x, y = x.to(device), y.to(device)
            pi, sigma, mu = mdn_model(x)
            loss = mdn_loss_fn(pi, sigma, mu, y)
    return -loss 
def feature_importance_perturbation(mdn_model, data_loader, device, perturbation_std=0.05):
    baseline_log_likelihood = compute_log_likelihood(mdn_model, data_loader, device)
    #print(f"Baseline Log-Likelihood: {baseline_log_likelihood}")
    num_features = data_loader.dataset[0][0].shape[0]
    importance_scores = np.zeros(num_features)
    for feature_idx in range(num_features):
        perturbed_log_likelihood = 0.0 
        for x, y in data_loader:
            x, y = x.to(device), y.to(device)
            perturbed_x = x.clone()
            noise = torch.normal(0, perturbation_std, size=(x.size(0),), device=device)
            perturbed_x[:, feature_idx] += noise  
            perturbed_x[:, feature_idx] = torch.clamp(perturbed_x[:, feature_idx], min=0.0, max=1.0)
            pi, sigma, mu = mdn_model(perturbed_x)
            loss = mdn_loss_fn(pi, sigma, mu, y)        
            perturbed_log_likelihood =-loss
            importance_scores[feature_idx] = torch.mean(torch.abs(baseline_log_likelihood - perturbed_log_likelihood))        
    return importance_scores 
X_test=np.load('xlo_test.npy')[:,:]
y_test=np.load('ylo_test.npy')[:,:] 
y_test[:,1]=y_test[:,1]+y_test[:,3]
y_test=y_test[:,1:2] #phase
test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32))
test_loader = DataLoader(test_dataset, batch_size=X_test.shape[0], shuffle=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MDN(n_hidden=83, n_gaussians=4,n_hidden_layers=7,bn=0, dout=0, dout_value=0.35831) #base BCC
model.eval()
optimizer = optim.Adam(model.parameters(), lr=0.00007)
PATH = "checkpoint/model-806.pt" 
checkpoint = torch.load(PATH)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
feature_importance_scores = feature_importance_perturbation(model, test_loader, device)
feature_ranking = np.argsort(np.abs(feature_importance_scores))[::-1]
#print("Feature Importance Ranking (Most Important to Least):")
#print(feature_ranking)  
feature_names = [
    r"$\Delta S_{\mathrm{mix}}$", r"$\Delta H_{\mathrm{mix}}$", r"$\Omega$", r"$\eta$", r"$K_1$", r"$\phi$", r"$\delta$",
    r"$\mathrm{VEC}$", r"$\Delta \chi$", r"$PFP_{\mathrm{FCC}}$", r"$PFP_{\mathrm{BCC}}$", r"$PFP_{\mathrm{HCP}}$", r"$PFP_{\mathrm{B2}}$",
    r"$PFP_{\mathrm{Laves}}$", r"$PFP_{\mathrm{Sigma}}$", r"$\mathrm{PSP}$", r"$\sigma(\Delta H_{\mathrm{mix}})$", r"$\mathrm{BulkModulus}$",
    r"$\sigma(\mathrm{BulkModulus})$", r"$T_m$", r"$\sigma_{T_m}$", r"$\sigma_{\mathrm{VEC}}$", r"$\chi$", r"$\mathrm{Atomic\ Number}$",
    r"$\mathrm{Group}$", r"$\mathrm{Families}$", r"$\mathrm{Quantum\ Number\ L}$", r"$\mathrm{Miracle\ Radius}$", r"$\mathrm{Covalent\ Radius}$", r"$\mathrm{Zunger\ Radius}$",
    r"$\mathrm{Ionic\ Radius}$", r"$\mathrm{Crystal\ Radius}$", r"$\mathrm{MB\ Electronegativity}$", r"$\mathrm{Gordy\ Electronegativity}$", r"$\mathrm{Allred{-}Rochow\ Electronegativity}$",
    r"$\mathrm{Polarizability}$", r"$\mathrm{Boiling\ Point}$", r"$\mathrm{Density}$", r"$\mathrm{Specific\ Heat}$", r"$\mathrm{Thermal\ Conductivity}$",
    r"$T$"
]
num_data_points=X_test.shape[0]
normalized_scores = [score / num_data_points for score in feature_importance_scores]
features = [f"{i}" for i in range(len(feature_importance_scores))]
sorted_indices = np.argsort(normalized_scores)[::-1]
sorted_features = [feature_names[i] for i in sorted_indices]
sorted_scores = [normalized_scores[i] for i in sorted_indices]
plt.figure(figsize=(8, 6.5))
plt.barh(sorted_features, sorted_scores, color="black")
plt.xlabel("Normalized Importance Score",fontsize=20, fontname='Times New Roman')
plt.ylabel("Feature name",fontsize=20, fontname='Times New Roman')
plt.gca().invert_yaxis()  
plt.xticks(fontsize=20, fontname='Times New Roman')
plt.yticks(fontsize=10, fontname='Times New Roman') 
plt.tight_layout()
#plt.savefig('6a.png', dpi=600)